In [1]:
import asyncio
import pythreejs as three
import pywavefront
from pythreejs import *
from ipywidgets import *
from IPython.display import display, Javascript
import numpy as np
import util








def _on_trans_slider(change):
    util.set_translation_global(cube, x_trans_slider.value, y_trans_slider.value, z_trans_slider.value)

def _on_rot_slider(change):
    util.set_rotation_global(cube, x_rot_slider.value, y_rot_slider.value, z_rot_slider.value, "XYZ")

def _on_scale_slider(change):
    util.set_scale_global(cube, x_scale_slider.value, y_scale_slider.value, z_scale_slider.value)



def update_cube_scale(x=1, y=1, z=1):
    cube.scale = (x,y,z)


    
def toggle_grid(change):
    grid_group.visible = not grid_group.visible

def toggle_axes(change):
    axes_group.visible = not axes_group.visible
    





def disable_sliders():
    x_trans_slider.unobserve(_on_trans_slider, names='value')
    y_trans_slider.unobserve(_on_trans_slider, names="value")
    z_trans_slider.unobserve(_on_trans_slider, names="value")
    x_rot_slider.unobserve(_on_rot_slider, names="value")
    y_rot_slider.unobserve(_on_rot_slider, names="value")
    z_rot_slider.unobserve(_on_rot_slider, names="value")
    x_scale_slider.unobserve(_on_scale_slider, names="value")
    y_scale_slider.unobserve(_on_scale_slider, names="value")
    z_scale_slider.unobserve(_on_scale_slider, names="value")

def enable_sliders():
    x_trans_slider.observe(_on_trans_slider, names='value')
    y_trans_slider.observe(_on_trans_slider, names="value")
    z_trans_slider.observe(_on_trans_slider, names="value")
    x_rot_slider.observe(_on_rot_slider, names="value")
    y_rot_slider.observe(_on_rot_slider, names="value")
    z_rot_slider.observe(_on_rot_slider, names="value")
    x_scale_slider.observe(_on_scale_slider, names="value")
    y_scale_slider.observe(_on_scale_slider, names="value")
    z_scale_slider.observe(_on_scale_slider, names="value")





#ich denke ich mache es so dass es eine synchrone und eine asynchrone funktion gibt. im besten fall nur zum debuggen aber ich hab kein gutes gefühl wegen dem zucken...
async def run():
    x = 0.003
    counter = 0
    disable_sliders()
    while(counter < 2000):
        util.set_translation_global(cube, counter, 0, 0)
        counter+=1
        await asyncio.sleep(0.01)
    enable_sliders()
    
    



width = 600
height = 400

cube_current_x = 0
cube_current_y = 0
cube_current_z = 0
scene = Scene()
scene.background = "#DDDDDD"

lock = asyncio.Lock()

fbx_model = pywavefront.Wavefront("C:/Users/Philipp/Desktop/gitProjects/visualkinematics/pythonAnsatz/assets/airboat.obj", collect_faces=True)  # Die 'model.obj' Datei laden
vertices = np.array(fbx_model.vertices)
#indices = np.array(fbx_model.meshes[0].faces).flatten()

indices = []
for name, mesh in fbx_model.meshes.items():
    indices.extend(mesh.faces)
indices = np.array(indices, dtype=np.uint32).flatten()

normals = util.compute_normals(vertices, indices)

fbx_geometry = three.BufferGeometry(
    attributes={
        'position': three.BufferAttribute(vertices, normalized=False),  # Positionsdaten
        'index': three.BufferAttribute(indices, normalized=False),  # Indices der Dreiecke
        'normal': three.BufferAttribute(normals, normalized=False),  # Hinzufügen der Normalen
    }
)


fbx_material = MeshStandardMaterial(color='orange')
fbx_mesh = three.Mesh(fbx_geometry, material=fbx_material)
fbx_mesh.scale = (0.05,0.05,0.05)



cube = util.createQuad(0,0,0,1,1,1)

# Licht und Kamera
light = PointLight(color='white', intensity=1.5, position=[5, 5, 5])
camera = PerspectiveCamera(position=[3, 3, 3],aspect=width/height, fov=50)

grid_group = util.create_grid(10,0.4)
axes_group = util.create_axes(4)

scene.add([camera, light, axes_group, grid_group, cube, AmbientLight(intensity=0.5)])





# Renderer mit Orbit-Steuerung
renderer = Renderer(camera=camera, scene=scene, controls=[OrbitControls(controlling=camera)], width=width, height=height, background_color="#87CEEB", background_opacity=1.0, antialias=True, precision='highp')



# Schieberegler
x_rot_slider = FloatSlider(min=-180, max=180, step=0.1, description='Rotate X')
y_rot_slider = FloatSlider(min=-180, max=180, step=0.1, description='Rotate Y')
z_rot_slider = FloatSlider(min=-180, max=180, step=0.1, description='Rotate Z')

x_scale_slider = FloatSlider(min=0, max=5, step=0.001, description="Scale X", value=1)
y_scale_slider = FloatSlider(min=0, max=5, step=0.001, description="Scale Y", value=1)
z_scale_slider = FloatSlider(min=0, max=5, step=0.001, description="Scale Z", value=1)

x_trans_slider = FloatSlider(min=0, max = 10, step=0.001, description="Translation X")
y_trans_slider = FloatSlider(min=0, max = 10, step=0.001, description="Translation Y")
z_trans_slider = FloatSlider(min=0, max = 10, step=0.001, description="Translation Z")

checkbox_grid = Checkbox(value=True, description='Show Grid')
checkbox_axes = Checkbox(value=True, description='Show Axes')


enable_sliders()
#interactive_control_scale = widgets.interactive(update_cube_scale, x=x_scale_slider, y=y_scale_slider, z=z_scale_slider)

checkbox_grid.observe(toggle_grid, names='value')
checkbox_axes.observe(toggle_axes, names='value')

trans_box = VBox([x_trans_slider, y_trans_slider, z_trans_slider])
rot_box = VBox([x_rot_slider, y_rot_slider, z_rot_slider])
scale_box = VBox([x_scale_slider, y_scale_slider, z_scale_slider])


display(renderer)
display(HBox([trans_box, rot_box, scale_box]))
display(HBox([checkbox_grid, checkbox_axes]))

c:\Users\Philipp\Desktop\gitProjects\visualkinematics\pythonAnsatz\util.py:34: RuntimeWarning: invalid value encountered in divide
  face_normal = face_normal / np.linalg.norm(face_normal)
c:\Users\Philipp\Desktop\gitProjects\visualkinematics\pythonAnsatz\.venv\lib\site-packages\pythreejs\traits.py:257: UserWarning: 64-bit data types not supported for WebGL data, casting to 32-bit.
  warnings.warn('64-bit data types not supported for WebGL '


Renderer(camera=PerspectiveCamera(aspect=1.5, position=(3.0, 3.0, 3.0), projectionMatrix=(1.0, 0.0, 0.0, 0.0, …